In [11]:
from glob import glob
import os
import pandas as pd
from pathlib import Path
import random
import stanza
# stanza.download("pt")
nlp = stanza.Pipeline("pt", download_method=None)

2026-06-07 19:54:57 INFO: Loading these models for language: pt (Portuguese):
| Processor    | Package         |
----------------------------------
| tokenize     | bosque          |
| mwt          | bosque          |
| pos          | bosque_charlm   |
| lemma        | bosque_nocharlm |
| constituency | cintil_charlm   |
| depparse     | bosque_charlm   |

2026-06-07 19:54:57 INFO: Using device: cuda
2026-06-07 19:54:57 INFO: Loading: tokenize
2026-06-07 19:54:57 INFO: Loading: mwt
2026-06-07 19:54:57 INFO: Loading: pos
2026-06-07 19:55:01 INFO: Loading: lemma
2026-06-07 19:55:02 INFO: Loading: constituency
2026-06-07 19:55:02 INFO: Loading: depparse
2026-06-07 19:55:03 INFO: Done loading processors!


In [ ]:
verbs_file = "./dataframes/verbs.csv"
verbs_df = pd.read_csv(verbs_file)

contractions_file = "./dataframes/contraction.csv"
contractions_df = pd.read_csv(contractions_file)

In [3]:
def process_contraction(token):
    text1 = token.words[0].text
    lemma1 = token.words[0].lemma
    
    candidates = contractions_df.loc[
        (contractions_df["lemma1"] == lemma1) &
        (contractions_df["text1"] != text1),
        "text"
    ].dropna().tolist()

    if not candidates:
        return token.text 
    
    rng = random.Random()
    new_word = rng.choice(candidates)
    return new_word

def process_verb(word):
    text = word.text
    lemma = word.lemma

    candidates = verbs_df.loc[
        (verbs_df["lemma"] == lemma) &
        (verbs_df["text"] != text),
        "text"
    ].dropna().tolist()

    if not candidates:
        return text

    rng = random.Random()
    new_word = rng.choice(candidates)
    return new_word

In [4]:
def mutate_token(token):
    words = token.words

    if len(words) > 1:
        return process_contraction(token)

    word = words[0]

    if word.pos == "PUNCT":
        return token.text

    if word.pos == "VERB":
        return process_verb(word)

    return token.text

def process_text(text):
    doc = nlp(text)

    pieces = []
    cursor = 0

    for sent in doc.sentences:
        for token in sent.tokens:
            start = token.start_char
            end = token.end_char

            pieces.append(text[cursor:start])
            pieces.append(mutate_token(token))

            cursor = end

    pieces.append(text[cursor:])

    return "".join(pieces)

In [5]:
def split_line_ending(line: str):
    """
    Separa o conteúdo da linha do seu final de linha.
    Preserva \n, \r\n ou \r.
    """
    if line.endswith("\r\n"):
        return line[:-2], "\r\n"
    elif line.endswith("\n"):
        return line[:-1], "\n"
    elif line.endswith("\r"):
        return line[:-1], "\r"
    else:
        return line, ""


def process_file(input_file, output_file, encoding="utf-8"):
    input_file = Path(input_file)
    output_file = Path(output_file)

    if input_file.resolve() == output_file.resolve():
        raise ValueError("O arquivo de entrada e saída não podem ser o mesmo.")

    with open(input_file, "r", encoding=encoding) as fin, \
         open(output_file, "w", encoding=encoding) as fout:

        for line in fin:
            content, line_ending = split_line_ending(line)

            # Linhas que começam com < ou --- não são modificadas
            if content.startswith("<") or content.startswith("---"):
                fout.write(content + line_ending)
                continue

            new_content = process_text(content)
            fout.write(new_content + line_ending)

In [13]:
src_folder = "../../../corpora/GeneratedCorpus/PB NATIVO/"
dst_folder = "../../../corpora/GeneratedCorpus/PB NATIVO - CORRUPTED/"

corpora = glob(src_folder + "*")

for i, corpus_file in enumerate(corpora):
    file_name = os.path.basename(corpus_file)
    
    print(f"{i+1}/{len(corpora)}: {file_name}")
    
    dst_file = os.path.join(dst_folder, f"corrupted_{file_name}")
    process_file(corpus_file, dst_file)

1/8: SUBCORPUS RELATAR PB NATIVO.txt
2/8: SUBCORPUS EXPLICAR PB NATIVO.txt
3/8: SUBCORPUS COMPARTILHAR PB NATIVO.txt
4/8: SUBCORPUS FAZER PB NATIVO.txt
5/8: SUBCORPUS RECRIAR PB NATIVO.txt
6/8: SUBCORPUS EXPLORAR PB NATIVO.txt
7/8: SUBCORPUS CAPACITAR PB NATIVO.txt
8/8: SUBCORPUS RECOMENDAR PB NATIVO.txt
